# WasteLens - Iteration 12 (Colab GPU)


Compute env is Colab GPU; protocol unchanged from docs/rejection_experiment/iteration12_distillation_protocol.md (registered pre-training, commit 3b88e70). Run every code cell top to bottom; do not skip or reorder.

## S0 - environment

In [1]:
!nvidia-smi


Tue Sep 29 17:39:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import sys, platform
import tensorflow as tf
print('Python:', sys.version.split()[0], '|', platform.platform())
print('TensorFlow:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs:', gpus if gpus else 'NONE - enable a GPU runtime')
assert gpus, 'STOP: no GPU visible'
import numpy, sklearn, PIL
print('numpy:', numpy.__version__,
      '| sklearn:', sklearn.__version__,
      '| pillow:', PIL.__version__)


Python: 3.13.15 | Linux-6.6.122+-x86_64-with-glibc2.39
TensorFlow: 2.20.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
numpy: 2.1.3 | sklearn: 1.6.1 | pillow: 11.3.0


## S1 - repo sync (start from main, pin registered commit)

In [3]:
import subprocess
from pathlib import Path
REPO = Path('/content/WasteLens')
PINNED = 'e8e13674072cf686ed3aa1ee2d5b4803e5536e79'
if not (REPO / 'src' / 'train_distill_adaptation.py').exists():
    subprocess.run(['git', 'clone',
                    'https://github.com/Anik00223/WasteLens.git',
                    str(REPO)], check=True)


In [4]:
%cd /content/WasteLens
!git fetch origin main
!git checkout main
!git rev-parse HEAD
!git rev-parse origin/main
!git status --short


/content/WasteLens
From https://github.com/Anik00223/WasteLens
 * branch            main       -> FETCH_HEAD
Already on 'main'
Your branch is up to date with 'origin/main'.
dd130c5caf11f806dfe195f0a6b38696312779d8
dd130c5caf11f806dfe195f0a6b38696312779d8


In [6]:
import subprocess
head = subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True,
                      text=True, cwd='/content/WasteLens').stdout.strip()
print('HEAD:', head)
assert head == 'dd130c5caf11f806dfe195f0a6b38696312779d8', \
    'STOP: HEAD != pinned commit'
print('repo pinned OK')


HEAD: dd130c5caf11f806dfe195f0a6b38696312779d8
repo pinned OK


## S2 - dataset + teacher locks

In [7]:
%cd /content/WasteLens
!ls scratch/adaptation_set | wc -l
!ls scratch/fresh_set | wc -l


/content/WasteLens
ls: cannot access 'scratch/adaptation_set': No such file or directory
0
ls: cannot access 'scratch/fresh_set': No such file or directory
0


In [9]:
from pathlib import Path

repo = Path("/content/WasteLens")
print("Repo:", repo)
print("Adaptation dataset exists:",
      (repo / "scratch/adaptation_set").exists())
print("Contents:")
for p in (repo / "scratch").glob("*"):
    print(p)

Repo: /content/WasteLens
Adaptation dataset exists: False
Contents:


## S3 - train seed 42 (6 epochs, FINAL epoch = candidate)

In [10]:
%cd /content/WasteLens
!python -u src/train_distill_adaptation.py --seed 42


/content/WasteLens
[head12] seed=42 epochs=6 lambda=0.5 T=2.0
Traceback (most recent call last):
  File "/content/WasteLens/src/train_distill_adaptation.py", line 473, in <module>
    main()
    ~~~~^^
  File "/content/WasteLens/src/train_distill_adaptation.py", line 370, in main
    locks = th.lock_dataset()
  File "/content/WasteLens/src/train_head_adaptation.py", line 84, in lock_dataset
    n_files = len([p for p in ADAPT_DIR.iterdir() if p.is_file()])
                              ~~~~~~~~~~~~~~~~~^^
  File "/usr/lib/python3.13/pathlib/_local.py", line 575, in iterdir
    with os.scandir(root_dir) as scandir_it:
         ~~~~~~~~~~^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'scratch/adaptation_set'


## S4 - train seed 43 (identical recipe)

In [11]:
%cd /content/WasteLens
!python -u src/train_distill_adaptation.py --seed 43


/content/WasteLens
[head12] seed=43 epochs=6 lambda=0.5 T=2.0
Traceback (most recent call last):
  File "/content/WasteLens/src/train_distill_adaptation.py", line 473, in <module>
    main()
    ~~~~^^
  File "/content/WasteLens/src/train_distill_adaptation.py", line 370, in main
    locks = th.lock_dataset()
  File "/content/WasteLens/src/train_head_adaptation.py", line 84, in lock_dataset
    n_files = len([p for p in ADAPT_DIR.iterdir() if p.is_file()])
                              ~~~~~~~~~~~~~~~~~^^
  File "/usr/lib/python3.13/pathlib/_local.py", line 575, in iterdir
    with os.scandir(root_dir) as scandir_it:
         ~~~~~~~~~~^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'scratch/adaptation_set'


## S5 - evaluate both seeds (threshold 0.0702, gates per protocol)

In [12]:
%cd /content/WasteLens
!python -u src/eval_head_adaptation.py --seed 42 --tag head12 --protocol iteration12_distillation_protocol.md --gates head12_gates.json
!python -u src/eval_head_adaptation.py --seed 43 --tag head12 --protocol iteration12_distillation_protocol.md --gates head12_gates.json


/content/WasteLens
candidate not found: models/checkpoints/wastelens_rej_head12s42_final.keras - train the seed first (src/train_head_adaptation.py --seed 42)
candidate not found: models/checkpoints/wastelens_rej_head12s43_final.keras - train the seed first (src/train_head_adaptation.py --seed 43)


In [13]:
%cd /content/WasteLens
!python -u src/eval_head_adaptation.py --summary --tag head12 --protocol iteration12_distillation_protocol.md


/content/WasteLens
no head12s{42,43}_gates.json - run --seed first


## S6 - evidence bundle (download to laptop)

In [16]:
%cd /content/WasteLens
!git status --short
!ls -la docs/rejection_experiment/head12*
!tar -czf /content/head12_evidence.tar.gz docs/rejection_experiment/head12* docs/rejection_experiment/iteration12_decision.md
!ls -la /content/head12_evidence.tar.gz


/content/WasteLens
ls: cannot access 'docs/rejection_experiment/head12*': No such file or directory
tar: docs/rejection_experiment/head12*: Cannot stat: No such file or directory
tar: docs/rejection_experiment/iteration12_decision.md: Cannot stat: No such file or directory
tar: Exiting with failure status due to previous errors
-rw-r--r-- 1 root root 45 Sep 29 17:45 /content/head12_evidence.tar.gz


In [15]:
from google.colab import files
files.download('/content/head12_evidence.tar.gz')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>